In [ ]:
import jax
import jax.numpy as jnp
from jax import lax, vmap, jit
from functools import partial
import jax.nn as jnn
import flax.linen as nn
from flax import struct
import optax
import gymnax
from gymnax.environments import environment, spaces
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
from typing import Tuple, NamedTuple, Any

# Auto-detect backend
backend = jax.default_backend()
is_gpu = backend == 'gpu'
print(f"JAX backend: {backend}")

# Global constants (Python ints, not JAX arrays)
T = 1e6           # Episode length
DELTA_T = 1e3       # Time duration between iteration to change intensity to probability
init_sample_dir = 'data/processed/init/init.parquet'

# Demo mode: reduce sweep size on CPU
DEMO_SIZE = 1 if is_gpu else 1  # Could be reduced further on CPU for testing

In [ ]:
@struct.dataclass
class MMEnvState(environment.EnvState):
    best_ask: jnp.ndarray
    best_bid: jnp.ndarray      
    q: jnp.ndarray      # inventory (int)
    spread: jnp.ndarray
    imb: jnp.ndarray
    ask_profile: jnp.ndarray
    bid_profile: jnp.ndarray
    # time is inherited from base EnvState

@struct.dataclass
class MMEnvParams(environment.EnvParams):
    T: int = T
    dt: float = DELTA_T
    init_sample_dir: str = init_sample_dir
    max_steps_in_episode: int = T

In [ ]:
class MMEnv(environment.Environment[MMEnvState, MMEnvParams]):
    '''Discrete market-making environment.'''

    @property
    def default_params(self):
        return MMEnvParams()

    def reset_env(self, key, init_samples, params):
        '''Reset environment to initial state.'''
        state = None
        # sample it from the init_samples

        obs = self.get_obs(state, params)
        return obs, state

    def step_env(self, key, state, action, params):
        '''Step environment: mid move, fills, reward.'''
        key_mid, key_fill = jax.random.split(key)

        # Decode action
        delta = action # decode of action should be included in model's parameter
        
        # place the order based on action
        state.ask_queue[action[0]] += 1
        state.bid_queue[action[1]] += 1
        
        

        q_post_liq = state.q + dq

        # Update state
        state_new = MMEnvState(
            S=S_new,
            q=q_post_liq,
            time=jnp.int32(state.time + 1)
        )
        
         # Terminal condition
        is_terminal = state.time >= params.max_steps_in_episode - 1
        
        # Reward calculation
        reward = reward(state, params)

        done = is_terminal
        info = {"filled_b": filled_b, "filled_a": filled_a}

        return (
            jax.lax.stop_gradient(self.get_obs(state_new, params)),
            jax.lax.stop_gradient(state_new),
            reward.astype(jnp.float32),
            done,
            info
        )

    def get_obs(self, state, params, key=None):
        """
        Remain for costomize
        """
        
        return None
    
    
    def reward(self, state, params, key=None):
        """
        Remain for costomize
        """
        
        return None
    
    def is_terminal(self, state, params):
        return state.time >= params.max_steps_in_episode

    @property
    def num_actions(self, params):
        return spaces.Discrete((params.K+1)**2)
        
    def action_space(self, params=None):
        # two quote levels, each in {0, ..., K}
        return spaces.Box(low=0, high=params.K, shape=(2,), dtype=jnp.int32)

    def observation_space(self, params=None):
        return spaces.Box(-jnp.inf, jnp.inf, (2,), jnp.float32)

    def state_space(self, params=None):
        # Need to modify
        return spaces.Dict({
            "S": spaces.Box(-jnp.inf, jnp.inf, (), jnp.float32),
            "q": spaces.Box(-jnp.inf, jnp.inf, (), jnp.int32),
            "time": spaces.Discrete(params.max_steps_in_episode),
        })

env = MMEnv()
params = MMEnvParams()